In [ ]:
import numpy as np
import pandas as pd
from aiwhatif_cf.config import DATA_DIR, CF_OUTPUTS

batch_cfcheck_data = CF_OUTPUTS / "tfnnc_random_2026-04-15" / "random_annotated_hltprhc.csv"
batch_cfcheck_df = pd.read_csv(batch_cfcheck_data)

In [11]:
help(pd.set_option)

Help on CallableDynamicDoc in module pandas._config.config:

<pandas._config.config.CallableDynamicDoc object>
    set_option(pat, value)
    
    Sets the value of the specified option.
    
    Available options:
    
    - compute.[use_bottleneck, use_numba, use_numexpr]
    - display.[chop_threshold, colheader_justify, date_dayfirst, date_yearfirst,
      encoding, expand_frame_repr, float_format]
    - display.html.[border, table_schema, use_mathjax]
    - display.[large_repr, max_categories, max_columns, max_colwidth, max_dir_items,
      max_info_columns, max_info_rows, max_rows, max_seq_items, memory_usage,
      min_rows, multi_sparse, notebook_repr_html, pprint_nest_depth, precision,
      show_dimensions]
    - display.unicode.[ambiguous_as_wide, east_asian_width]
    - display.[width]
    - future.[infer_string, no_silent_downcasting]
    - io.excel.ods.[reader, writer]
    - io.excel.xls.[reader]
    - io.excel.xlsb.[reader]
    - io.excel.xlsm.[reader, writer]
    - io.ex

In [12]:
pd.set_option("display.max_rows", None)

In [13]:
batch_cfcheck_df = batch_cfcheck_df.drop(columns=["hltprhc", "target_risk"])

In [14]:

batch_cfcheck_df["cf_id"] = batch_cfcheck_df["cf_id"].replace({"original": "xin"})

batch_cfcheck_df = batch_cfcheck_df.rename(columns={
    "original_risk": "risk_before",
    "predicted_risk" : "predicted_risk_after",
    "meets_target_risk" : "valid",
})

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid
0,0,xin,2.0,3.0,6.0,3.0,1.0,0.0,20.438166,2.0,0.503426,NaN,NaN
1,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,20.580482,4.0,0.503426,0.495227,False
2,0,cf_2,NaN,NaN,3.0,NaN,NaN,NaN,20.283558,NaN,0.503426,0.498998,False
3,0,cf_3,NaN,NaN,NaN,NaN,NaN,NaN,22.613884,7.0,0.503426,0.491371,False


In [15]:
int_columns = [
    "etfruit",
    "eatveg",
    "cgtsmok",
    "alcfreq",
    "slprl",
    "paccnois",
    "dosprt",
]

float_columns = [
    "bmi",
    "risk_before",
    "predicted_risk_after",
]

In [22]:
for col in int_columns:
    batch_cfcheck_df[col] = (
        pd.to_numeric(batch_cfcheck_df[col], errors="coerce")
        .round()          # <-- detta är nyckeln
        .astype("Int64")
    )


In [ ]:
batch_cfcheck_df[float_columns] = batch_cfcheck_df[float_columns].round(2)

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid
0,0,xin,2,3,6,3,1,0,20.44,2,0.06,NaN,NaN
1,0,cf_1,<NA>,<NA>,<NA>,<NA>,4,<NA>,21.52,<NA>,0.06,0.02,True
2,0,cf_2,4,<NA>,<NA>,<NA>,<NA>,<NA>,24.45,<NA>,0.06,0.04,False
3,0,cf_3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,22.92,<NA>,0.06,0.08,False


In [23]:
batch_cfcheck_df[int_columns] = (
    batch_cfcheck_df[int_columns]
    .astype("string")
    .fillna("")
)

In [24]:
batch_cfcheck_df = batch_cfcheck_df.replace({np.nan : ""})
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid
0,0,xin,2,3,6,3,1,0,20.438166,2,0.503426,,
1,0,cf_1,,,,,,,20.580482,4,0.503426,0.495227,False
2,0,cf_2,,,3,,,,20.283558,,0.503426,0.498998,False
3,0,cf_3,,,,,,,22.613884,7,0.503426,0.491371,False


In [25]:
mask = batch_cfcheck_df["cf_id"] == "xin"
batch_cfcheck_df.loc[mask, "risk_before"] = ""
batch_cfcheck_df.head(4)

/tmp/ipykernel_39799/1160248817.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  batch_cfcheck_df.loc[mask, "risk_before"] = ""


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid
0,0,xin,2,3,6,3,1,0,20.438166,2,,,
1,0,cf_1,,,,,,,20.580482,4,0.503426,0.495227,False
2,0,cf_2,,,3,,,,20.283558,,0.503426,0.498998,False
3,0,cf_3,,,,,,,22.613884,7,0.503426,0.491371,False


In [26]:
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# 1. Beräkna Nchanged endast för CF-rader
mask_cf = batch_cfcheck_df["cf_id"] != "xin"

batch_cfcheck_df.loc[mask_cf, "Nchanged"] = (
    batch_cfcheck_df.loc[mask_cf, feature_cols] != ""
).sum(axis=1)

# 2. Konvertera kolumnen till ren string
batch_cfcheck_df["Nchanged"] = batch_cfcheck_df["Nchanged"].astype("string")

# 3. Baseline ska vara tom
batch_cfcheck_df.loc[batch_cfcheck_df["cf_id"] == "xin", "Nchanged"] = ""

In [27]:
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,risk_before,predicted_risk_after,valid,Nchanged
0,0,xin,2,3,6,3,1,0,20.438166,2,,,,
1,0,cf_1,,,,,,,20.580482,4,0.503426,0.495227,False,2
2,0,cf_2,,,3,,,,20.283558,,0.503426,0.498998,False,2
3,0,cf_3,,,,,,,22.613884,7,0.503426,0.491371,False,2


In [28]:
batch_cfcheck_df["Gower"] = ""
batch_cfcheck_df["Expected"] = ""


In [29]:
order = ["query_index", "cf_id"] + feature_cols + ["Gower", "Nchanged", "valid", "risk_before", "predicted_risk_after", "Expected"]

batch_cfcheck_df = batch_cfcheck_df[order]
batch_cfcheck_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,2,3,6,3,1,0,20.438166,2,,,,,,
1,0,cf_1,,,,,,,20.580482,4,,2,False,0.503426,0.495227,
2,0,cf_2,,,3,,,,20.283558,,,2,False,0.503426,0.498998,
3,0,cf_3,,,,,,,22.613884,7,,2,False,0.503426,0.491371,
4,1,xin,3,4,1,2,3,0,22.37568,0,,,,,,
5,1,cf_1,,,,4,,,,,,1,False,0.501517,0.493254,
6,1,cf_2,,,,,,,25.075153,,,1,False,0.501517,0.499514,
7,1,cf_3,,2,,,4,,,,,2,False,0.501517,0.493271,
8,2,xin,5,3,1,1,4,0,22.694019,7,,,,,,
9,2,cf_1,,,,,,,22.613884,,,1,False,0.493596,0.49358,


# picking correct CF

In [30]:
# --- Select exactly one CF per query_index (first valid, else first) ---

# Split baseline and CF rows
df_xin = batch_cfcheck_df[batch_cfcheck_df["cf_id"] == "xin"]
df_cf  = batch_cfcheck_df[batch_cfcheck_df["cf_id"] != "xin"]

# Sort CF rows so valid=True comes first within each query_index
df_cf_sorted = df_cf.sort_values(
    ["query_index", "valid"],
    ascending=[True, False]   # valid=True first
)

# Select one CF per query_index (first after sorting)
df_cf_selected = df_cf_sorted.groupby("query_index", as_index=False).first()

# Combine baseline + selected CF
batch_cfcheck_df = pd.concat([df_xin, df_cf_selected], ignore_index=True)

# Ensure xin appears before CF for each query_index
batch_cfcheck_df["is_xin"] = (batch_cfcheck_df["cf_id"] == "xin").astype(int)
batch_cfcheck_df = (
    batch_cfcheck_df
    .sort_values(["query_index", "is_xin"], ascending=[True, False])
    .drop(columns="is_xin")
)
batch_cfcheck_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,2,3,6,3,1,0,20.438166,2,,,,,,
9,0,cf_1,,,,,,,20.580482,4,,2,False,0.503426,0.495227,
1,1,xin,3,4,1,2,3,0,22.37568,0,,,,,,
10,1,cf_1,,,,4,,,,,,1,False,0.501517,0.493254,
2,2,xin,5,3,1,1,4,0,22.694019,7,,,,,,
11,2,cf_1,,,,,,,22.613884,,,1,False,0.493596,0.49358,
3,3,xin,6,4,1,1,1,0,26.014568,0,,,,,,
12,3,cf_1,,2,,,4,,,,,2,False,0.531336,0.499009,
4,4,xin,2,3,5,4,1,0,27.681661,4,,,,,,
13,4,cf_1,,,,,,,23.065737,,,1,False,0.503134,0.495573,


In [31]:
batch_cfcheck_df["valid"] = batch_cfcheck_df["valid"].replace(
    {
        False : "No",
        True : "Yes",
    }
)

In [32]:
# ensure BMI is numeric so comparisons work
batch_cfcheck_df["bmi"] = pd.to_numeric(batch_cfcheck_df["bmi"], errors="coerce")

# extract baseline BMI per query_index
baseline_bmi = batch_cfcheck_df[batch_cfcheck_df["cf_id"] == "xin"] \
    .set_index("query_index")["bmi"]

# compute Expected
def compute_expected(row):
    if row["cf_id"] == "xin":
        return ""
    bmi_xin = baseline_bmi.loc[row["query_index"]]
    return "NO" if row["bmi"] > bmi_xin else ""

batch_cfcheck_df["Expected"] = batch_cfcheck_df.apply(compute_expected, axis=1)


In [33]:
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,2,3,6,3,1,0,20.438166,2,,,,,,
9,0,cf_1,,,,,,,20.580482,4,,2,No,0.503426,0.495227,NO
1,1,xin,3,4,1,2,3,0,22.375680,0,,,,,,
10,1,cf_1,,,,4,,,NaN,,,1,No,0.501517,0.493254,


In [34]:
batch_cfcheck_df = batch_cfcheck_df.drop(columns=["query_index"])
batch_cfcheck_df

,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,Gower,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,xin,2,3,6,3,1,0,20.438166,2,,,,,,
9,cf_1,,,,,,,20.580482,4,,2,No,0.503426,0.495227,NO
1,xin,3,4,1,2,3,0,22.375680,0,,,,,,
10,cf_1,,,,4,,,NaN,,,1,No,0.501517,0.493254,
2,xin,5,3,1,1,4,0,22.694019,7,,,,,,
11,cf_1,,,,,,,22.613884,,,1,No,0.493596,0.49358,
3,xin,6,4,1,1,1,0,26.014568,0,,,,,,
12,cf_1,,2,,,4,,NaN,,,2,No,0.531336,0.499009,
4,xin,2,3,5,4,1,0,27.681661,4,,,,,,
13,cf_1,,,,,,,23.065737,,,1,No,0.503134,0.495573,
